In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, LayerNormalization, Input, Concatenate
from tensorflow.keras.models import Model
import numpy as np

In [3]:
Conjunctiva_features = np.load("Features/Conjunctiva_train.npy")
#features_train_palm = np.load("dense_features_train_palm.npy")
#features_train_nail = np.load("dense_features_train_nail.npy")
Textual_features = np.load("Features/text_train.npy")


In [4]:
Conjunctiva_features = tf.convert_to_tensor(Conjunctiva_features, dtype=tf.float32)
Textual_features = tf.convert_to_tensor(Textual_features, dtype=tf.float32)
#palm_train = tf.convert_to_tensor(features_train_palm, dtype=tf.float32)
#nail_train = tf.convert_to_tensor(features_train_nail, dtype=tf.float32)


In [5]:
Conjunctiva_features = tf.expand_dims(Conjunctiva_features, axis=1)
Textual_features = tf.expand_dims(Textual_features, axis=1)

# Self Attention

In [6]:
def self_attention_block(x, num_heads=4):
    d_model = x.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(x, x)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(x + attn)
    return out

In [10]:
conj_input = Input(shape=(1, Conjunctiva_features.shape[-1]), name="Conjunctiva_Input")
text_input = Input(shape=(1, Textual_features.shape[-1]), name="Text_Input")
conj_self = self_attention_block(conj_input)
text_self = self_attention_block(text_input)

# Cross Attention

In [7]:
def cross_attention_block(query, key_value, num_heads=4):
    d_model = query.shape[-1]
    
    attn = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads
    )(query, key_value)
    
    attn = Dropout(0.2)(attn)
    out = LayerNormalization()(query + attn)
    return out


In [11]:
text_cross_conj = cross_attention_block(text_self, conj_self)

In [12]:
from tensorflow.keras.layers import Flatten

conj_flat = Flatten()(conj_self)
text_flat = Flatten()(text_self)
text_cross_flat = Flatten()(text_cross_conj)

# Concatenate
fusion_vector = Concatenate()([conj_flat, text_flat, text_cross_flat])


In [13]:
x = Dense(512, activation='relu')(fusion_vector)
x = Dropout(0.5)(x)

output_class = Dense(1, activation='sigmoid', name="Anemia_Class")(x)  # classification
output_hb = Dense(1, activation='linear', name="Hemoglobin")(x)        # regression


In [14]:
final_model = Model(
    inputs=[conj_input, text_input],
    outputs=[output_class, output_hb]
)

final_model.compile(
    optimizer='adam',
    loss={'Anemia_Class':'binary_crossentropy', 'Hemoglobin':'mse'},
    metrics={'Anemia_Class':'accuracy', 'Hemoglobin':'mae'}
)

final_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Conjunctiva_Input   │ (None, 1, 1024)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Input          │ (None, 1, 32)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 1024)   │  4,198,400 │ Conjunctiva_Inpu… │
│ (MultiHeadAttentio… │                   │            │ Conjunctiva_Inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │      4,224 │ Text_Input[0][0], │
│ (MultiHeadAttentio… │                   │            │ Text_Input[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1, 1024)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 1, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1, 1024)   │          0 │ Conjunctiva_Inpu… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 1, 32)     │          0 │ Text_Input[0][0], │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 1, 1024)   │      2,048 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 32)     │     67,712 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 1, 32)     │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 1, 32)     │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 32)     │         64 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1024)      │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 32)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1088)      │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,831,106 (18.43 MB)

 Trainable params: 4,831,106 (18.43 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
# Assuming y_train_class (0/1) and y_train_hb (float)
final_model.fit(
    [Conjunctiva_features, Textual_features],
    [y_train_class, y_train_hb],
    validation_data=([Conj_val, Text_val], [y_val_class, y_val_hb]),
    epochs=30,
    batch_size=16
)


NameError: name 'y_train_class' is not defined